# 18 - Hybrid Algorithms & Parameter-Shift Gradients

Compute quantum gradients using the parameter-shift rule.

**Concepts:** Parameter-shift rule, analytic gradients, gradient descent

In [ ]:
import quantsdk as qs
import math
import numpy as np

## Parameter-Shift Rule

For a gate $R(\theta)$, the gradient is:

$$\frac{\partial \langle H \rangle}{\partial \theta} = \frac{1}{2}\left[\langle H \rangle_{\theta+\pi/2} - \langle H \rangle_{\theta-\pi/2}\right]$$

In [ ]:
def parameterized_circuit(theta: float) -> qs.Circuit:
    """Simple circuit: RY(theta)|0>, measure Z."""
    circuit = qs.Circuit(1, name="param")
    circuit.ry(0, theta)
    circuit.measure(0)
    return circuit

def expectation_z(theta: float, shots: int = 4000) -> float:
    """Measure <Z> = P(0) - P(1) = cos(theta)."""
    circuit = parameterized_circuit(theta)
    result = qs.run(circuit, shots=shots, seed=42)
    p0 = result.counts.get('0', 0) / shots
    p1 = result.counts.get('1', 0) / shots
    return p0 - p1

def parameter_shift_gradient(theta: float, shots: int = 4000) -> float:
    """Compute gradient using parameter-shift rule."""
    shift = math.pi / 2
    forward = expectation_z(theta + shift, shots)
    backward = expectation_z(theta - shift, shots)
    return (forward - backward) / 2

In [ ]:
# Compare quantum gradient with exact
print("Theta    | Quantum Grad | Exact Grad (-sin(theta))")
print("-" * 52)
for theta in np.linspace(0, 2 * math.pi, 9):
    q_grad = parameter_shift_gradient(theta)
    exact_grad = -math.sin(theta)
    print(f"{theta:6.3f}   | {q_grad:+.4f}      | {exact_grad:+.4f}")

## Gradient Descent to Minimize <Z>

In [ ]:
# Minimize <Z> = cos(theta) -> minimum at theta = pi
theta = 0.5  # Start near 0
lr = 0.3
history = []

for step in range(20):
    exp_val = expectation_z(theta)
    grad = parameter_shift_gradient(theta)
    history.append((theta, exp_val))
    theta -= lr * grad  # Gradient descent

print("Gradient Descent Progress:")
for i, (t, e) in enumerate(history):
    if i % 4 == 0:
        bar = '#' * int((e + 1) * 20)
        print(f"  Step {i:2d}: theta={t:6.3f}, <Z>={e:+.3f} {bar}")

print(f"\nFinal theta: {theta:.3f} (optimal: {math.pi:.3f})")
print(f"Final <Z>: {expectation_z(theta):.3f} (optimal: -1.0)")

## Multi-Parameter Gradient

In [ ]:
def two_param_circuit(t0: float, t1: float) -> float:
    """Circuit with two RY gates, measure <Z0>."""
    circuit = qs.Circuit(2, name="2param")
    circuit.ry(0, t0).ry(1, t1).cx(0, 1).measure(0)
    result = qs.run(circuit, shots=4000, seed=42)
    p0 = result.counts.get('0', 0) / 4000
    p1 = result.counts.get('1', 0) / 4000
    return p0 - p1

# Gradient with respect to t0
t0, t1 = 1.0, 0.5
shift = math.pi / 2

grad_t0 = (two_param_circuit(t0 + shift, t1) - two_param_circuit(t0 - shift, t1)) / 2
grad_t1 = (two_param_circuit(t0, t1 + shift) - two_param_circuit(t0, t1 - shift)) / 2

print(f"Parameters: t0={t0:.2f}, t1={t1:.2f}")
print(f"Gradient: (d/dt0={grad_t0:+.4f}, d/dt1={grad_t1:+.4f})")